In [ ]:
# 1. Install the necessary libraries for Qwen3-VL and quantization
!pip install -q git+https://github.com/huggingface/transformers.git

# THE FIX: Added -U to force upgrade, and explicitly set bitsandbytes>=0.46.1
!pip install -qU datasets accelerate peft "bitsandbytes>=0.46.1" trl qwen-vl-utils

In [3]:
from datasets import load_dataset

model_id = "Qwen/Qwen3-VL-4B-Instruct"

print("Loading modern Parquet version of TextVQA...")
# Using the pre-converted parquet dataset which bypasses the script error
# Loads all 34,600+ rows
dataset = load_dataset("lmms-lab/textvqa", split="train")

print(f"Dataset loaded! Number of rows: {len(dataset)}")
print(f"Target Model: {model_id}")

Loading modern Parquet version of TextVQA...


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

data/train-00000-of-00020.parquet:   0%|          | 0.00/303M [00:00<?, ?B/s]

...

data/test-00003-of-00004.parquet:   0%|          | 0.00/238M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/34602 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5734 [00:00<?, ? examples/s]

Dataset loaded! Number of rows: 2000
Target Model: Qwen/Qwen3-VL-4B-Instruct


In [4]:
# Select the very first example from our loaded dataset
sample = dataset[0]

# Print out the columns available in this dataset
print("Dataset Columns:", sample.keys())
print("-" * 50)

# Print the question
print("Question:", sample['question'])

# Print the ground-truth answers (TextVQA usually provides multiple human answers)
print("Answers List:", sample['answers'])

# Print the image details (without rendering the whole image yet)
print("Image Type/Size:", type(sample['image']), sample['image'].size)

Dataset Columns: dict_keys(['image_id', 'question_id', 'question', 'question_tokens', 'image', 'image_width', 'image_height', 'flickr_original_url', 'flickr_300k_url', 'answers', 'image_classes', 'set_name', 'ocr_tokens'])
--------------------------------------------------
Question: what is the brand of phone?
Answers List: ['nokia', 'nokia', 'nokia', 'nokia', 'toshiba', 'nokia', 'nokia', 'nokia', 'nokia', 'nokia']
Image Type/Size: <class 'PIL.JpegImagePlugin.JpegImageFile'> (1024, 730)


In [6]:
from collections import Counter

def prepare_vqa_item(example):
    # 1. Find the most frequent answer for high accuracy
    answer_counts = Counter(example["answers"])
    best_answer = answer_counts.most_common(1)[0][0]
    
    # 2. Keep text instructions in messages (without nesting the raw image here)
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": f"Question: {example['question']}\nAnswer the question concisely."}
            ]
        },
        {
            "role": "assistant",
            "content": [
                {"type": "text", "text": best_answer}
            ]
        }
    ]
    
    # 3. Return a clean structure: text in messages, image in its own dedicated field
    return {
        "messages": messages,
        "image": example["image"]
    }

print("Formatting dataset cleanly...")
# Re-run map with the decoupled structure
formatted_dataset = dataset.map(prepare_vqa_item, remove_columns=dataset.column_names)

print("\nFormatting complete without errors!")
print("Sample keys available now:", formatted_dataset[0].keys())

Formatting dataset cleanly...


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]


Formatting complete without errors!
Sample keys available now: dict_keys(['image', 'messages'])


In [8]:
import torch
from transformers import BitsAndBytesConfig, AutoProcessor, AutoModelForImageTextToText

# 1. Configure 4-bit quantization to save VRAM
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",          # NormalFloat 4, optimal for data precision
    bnb_4bit_compute_dtype=torch.float16, # Performs internal math in 16-bit for speed
    bnb_4bit_use_double_quant=True      # Quantizes the constants for extra memory savings
)

print("Loading Qwen3-VL Processor...")
processor = AutoProcessor.from_pretrained(model_id)

print("Loading Qwen3-VL 4B Model in 4-bit precision... (This may take 2-3 mins)")
# Using the updated class name for Transformers 5.0+
model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto" # Automatically splits layers across available T4 GPUs
)

print("\nProcessor and quantized Model loaded successfully!")

Loading Qwen3-VL Processor...


preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/713 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]


Processor and quantized Model loaded successfully!


In [9]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# 1. Prepare the quantized model for training (handles gradient checkpointing setups safely)
model = prepare_model_for_kbit_training(model)

# 2. Configure the LoRA parameters
lora_config = LoraConfig(
    r=16,                               # Rank: dimension of the low-rank matrices
    lora_alpha=32,                      # Scaling factor for the adapter weights
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"], # Targets the attention layers
    lora_dropout=0.05,                  # Regularization to prevent overfitting
    bias="none",                        # No bias parameters trained
    task_type="CAUSAL_LM"               # Vision-language models generate text tokens autoregressively
)

# 3. Wrap the base model with the LoRA layers
print("Applying LoRA adapters to the model...")
model = get_peft_model(model, lora_config)

# 4. Verify the reduction in trainable parameters
model.print_trainable_parameters()

Applying LoRA adapters to the model...
trainable params: 11,796,480 || all params: 4,449,612,288 || trainable%: 0.2651


In [10]:
import torch

# Ensure the tokenizer has a padding token defined (Qwen sometimes defaults to None)
if processor.tokenizer.pad_token_id is None:
    processor.tokenizer.pad_token_id = processor.tokenizer.eos_token_id

def vqa_data_collator(examples):
    # 1. Format the text conversation using the model's internal chat template
    texts = [
        processor.apply_chat_template(example["messages"], tokenize=False) 
        for example in examples
    ]
    
    # 2. Extract the raw images
    images = [example["image"] for example in examples]
    
    # 3. Pass both to the processor to create the mathematical tensors (input_ids, pixel_values)
    batch = processor(
        text=texts,
        images=images,
        padding=True,
        return_tensors="pt"
    )
    
    # 4. Create the labels for training
    labels = batch["input_ids"].clone()
    
    # High Accuracy Tip: Mask padding tokens with -100 so the model is not penalized for blank spaces
    labels[labels == processor.tokenizer.pad_token_id] = -100
    
    batch["labels"] = labels
    
    return batch

print("Custom Vision-Language Data Collator created successfully!")

Custom Vision-Language Data Collator created successfully!


In [11]:
from transformers import Trainer, TrainingArguments

print("Configuring Training Arguments with Kaggle guards...")

training_args = TrainingArguments(
    output_dir="./qwen3_vqa_results",
    num_train_epochs=1,                 # Still just 1 epoch!
    per_device_train_batch_size=2,      
    gradient_accumulation_steps=4,      
    learning_rate=2e-4,                 
    fp16=True,                          
    logging_steps=50,                   # Update logs less frequently to save output space
    
    # --- THE CRITICAL CHANGES FOR LARGE DATASETS ---
    save_strategy="steps",
    save_steps=500,                     # Save a checkpoint roughly every 45-60 minutes
    save_total_limit=2,                 # Still keeps only 2 to protect the 19GB limit!
    remove_unused_columns=False,        
    report_to="none"                    
)

# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=formatted_dataset,
    data_collator=vqa_data_collator,    # Our custom processor-driven collator from Step 6
)

print("\nTrainer successfully initialized and guarded against storage limits!")
print("Ready to begin training loop.")

Configuring Training Arguments with Kaggle guards...

Trainer successfully initialized and guarded against storage limits!
Ready to begin training loop.


In [13]:
from collections import Counter
from transformers import Trainer

print("🔧 Applying the Image Placeholder Fix...")

def prepare_vqa_item_fixed(example):
    answer_counts = Counter(example["answers"])
    best_answer = answer_counts.most_common(1)[0][0]
    
    messages = [
        {
            "role": "user",
            "content": [
                # THE FIX: We put the image flag back so the text gets the <|image_pad|> tokens, 
                # but we intentionally do NOT attach the raw PIL image here.
                {"type": "image"}, 
                {"type": "text", "text": f"Question: {example['question']}\nAnswer the question concisely."}
            ]
        },
        {
            "role": "assistant",
            "content": [
                {"type": "text", "text": best_answer}
            ]
        }
    ]
    
    return {
        "messages": messages,
        "image": example["image"] # The raw image is safely passed alongside
    }

# 1. Remap the dataset with the fixed placeholders
formatted_dataset = dataset.map(prepare_vqa_item_fixed, remove_columns=dataset.column_names)

# 2. Re-initialize the Trainer with the newly fixed dataset
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=formatted_dataset,
    data_collator=vqa_data_collator,
)

print("✅ Fix applied! Launching QLoRA Fine-Tuning...\n")

# 3. Launch the training loop
trainer.train()

print("\n🎉 Training complete! Saving final LoRA adapters...")
trainer.save_model("./qwen3_vqa_final_adapter")
processor.save_pretrained("./qwen3_vqa_final_adapter")
print("💾 Model successfully trained and saved!")

🔧 Applying the Image Placeholder Fix...


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

✅ Fix applied! Launching QLoRA Fine-Tuning...



[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: The AccumulateGrad node's stream does not match the stream of the node that produced the incoming gradient. This may incur unnecessary synchronization and break CUDA graph capture if the AccumulateGrad node's stream is the default stream. This mismatch is caused by an AccumulateGrad node created prior to the current iteration being kept alive. This can happen if the autograd graph is still being kept alive by tensors such as the loss, or if you are using DDP, which will stash a reference to the node. To resolve the mismatch, delete all references to the autograd graph or ensure that DDP initialization is performed under the same stream as subsequent forwards. If the mismatch is intentional, you can use torch.autograd.graph.set_warn_on_accumulate_grad_stream_mismatch(False) to suppress this warning. (Tri

Step,Training Loss
10,15.607901
20,8.309370
30,7.594814
40,7.497916
50,7.476021
60,7.464552
70,7.448315
80,7.450414
90,7.445530
100,7.443819


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the


🎉 Training complete! Saving final LoRA adapters...
💾 Model successfully trained and saved!


In [18]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText
from peft import PeftModel

print("Loading Base Model and Your Trained Adapters...")

# 1. Load the base model (instant from cache)
base_model_id = "Qwen/Qwen3-VL-4B-Instruct"
base_model = AutoModelForImageTextToText.from_pretrained(
    base_model_id,
    device_map="auto",
    torch_dtype=torch.float16
)

# 2. Attach your freshly trained LoRA adapters
adapter_path = "./qwen3_vqa_final_adapter"
model = PeftModel.from_pretrained(base_model, adapter_path)
processor = AutoProcessor.from_pretrained(adapter_path)

print("Model ready! Pulling a test sample directly from your local dataset memory...")

# 3. Securely grab an image and question from your loaded dataset
# We'll grab index 500 as an example test sample
sample_index = 500
test_image = dataset[sample_index]["image"]
question = dataset[sample_index]["question"]
ground_truth_answers = dataset[sample_index]["answers"]

# 4. Format the prompt exactly how we trained it
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": f"Question: {question}\nAnswer the question concisely."}
        ]
    }
]

# 5. Process the inputs for the GPU
text_prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = processor(text=[text_prompt], images=[test_image], padding=True, return_tensors="pt")
inputs = inputs.to("cuda")

print(f"\nTarget Question: {question}")
print("Thinking...")

# 6. Generate the answer!
with torch.no_grad():
    generated_ids = model.generate(**inputs, max_new_tokens=20)

# 7. Decode the output tokens
generated_ids_trimmed = [
    out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
output_text = processor.batch_decode(generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]

print("\n--- RESULTS ---")
print(f"🤖 Model Predicted Answer: {output_text.strip()}")
print(f"📝 Dataset Dataset Answers: {list(set(ground_truth_answers))}")

Loading Base Model and Your Trained Adapters...


Loading weights:   0%|          | 0/713 [00:00<?, ?it/s]

Model ready! Pulling a test sample directly from your local dataset memory...

Target Question: what airline is the plane on the back?
Thinking...

--- RESULTS ---
🤖 Model Predicted Answer: ryanair
📝 Dataset Dataset Answers: ['ryanair', 'extended', 'yes']


In [19]:
import ipywidgets as widgets
from IPython.display import display, clear_output
from PIL import Image
import io
import torch

print("🎨 Building Interactive VQA Interface...")

# 1. Create the UI widgets
upload_widget = widgets.FileUpload(
    accept='image/*',  # Accept all image types
    multiple=False,    # Single file only
    description='Upload Image',
    button_style='info'
)

question_widget = widgets.Text(
    value='What is in this image?',
    placeholder='Type your question here...',
    description='Question:',
    layout=widgets.Layout(width='50%')
)

run_button = widgets.Button(
    description='Ask Model',
    button_style='success',
    tooltip='Click to run the model'
)

output_area = widgets.Output()

# 2. Define what happens when the button is clicked
def on_button_click(b):
    with output_area:
        clear_output(wait=True) # Clear previous results
        
        # Check if an image is uploaded
        if not upload_widget.value:
            print("⚠️ Please upload an image first!")
            return
            
        print("Processing image...")
        
        # Safely extract the image data (handles different ipywidgets versions)
        try:
            # For modern ipywidgets (v8+)
            content = upload_widget.value[0].content
        except AttributeError:
            # For older ipywidgets (v7)
            content = list(upload_widget.value.values())[0]['content']
            
        # Convert bytes to a PIL Image
        raw_image = Image.open(io.BytesIO(content)).convert("RGB")
        
        # Display a smaller preview of the image so it fits nicely on screen
        display_img = raw_image.copy()
        display_img.thumbnail((400, 400))
        display(display_img)
        
        question = question_widget.value
        print(f"\nQuestion: {question}")
        print("Thinking...")
        
        # Format the prompt
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image"},
                    {"type": "text", "text": f"Question: {question}\nAnswer the question concisely."}
                ]
            }
        ]
        
        # Process the inputs
        text_prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = processor(text=[text_prompt], images=[raw_image], padding=True, return_tensors="pt")
        inputs = inputs.to("cuda")
        
        # Generate the answer
        with torch.no_grad():
            generated_ids = model.generate(**inputs, max_new_tokens=20)
            
        generated_ids_trimmed = [
            out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]
        output_text = processor.batch_decode(generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]
        
        print("\n--- RESULTS ---")
        print(f"🤖 Model Answer: {output_text.strip()}")

# 3. Connect the button to the function
run_button.on_click(on_button_click)

# 4. Display the interface
ui = widgets.VBox([
    widgets.HBox([upload_widget, question_widget]), 
    run_button, 
    output_area
])
display(ui)

🎨 Building Interactive VQA Interface...


In [20]:
import shutil
from IPython.display import FileLink, display

# 1. Define the folder we want to zip and the output name
folder_to_zip = "./qwen3_vqa_final_adapter"
zip_filename = "my_qwen3_vqa_model"

print("🗜️ Compressing model files into a zip archive... (This may take a minute)")

# 2. Create the zip file
shutil.make_archive(zip_filename, 'zip', folder_to_zip)

# 3. Generate a clickable download link
print("✅ Compression complete! Click the link below to download your model to your computer:")
display(FileLink(f"{zip_filename}.zip"))

🗜️ Compressing model files into a zip archive... (This may take a minute)
✅ Compression complete! Click the link below to download your model to your computer:


/kaggle/working/my_qwen3_vqa_model.zip